In [ ]:
!pip install datasets kagglehub -q


In [ ]:
import kagglehub
path = kagglehub.dataset_download("ishanikathuria/handwritten-signature-datasets")
print("Path to dataset files:", path)


100%|██████████| 290M/290M [00:03<00:00, 90.5MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/ishanikathuria/handwritten-signature-datasets/versions/3


In [ ]:
import os
os.kill(os.getpid(), 9)


In [ ]:
import torch
print("PyTorch version:", torch.__version__)
print("GPU Active:", torch.cuda.is_available())


In [ ]:
# ============================================================
# HANDWRITTEN SIGNATURE VERIFICATION USING ML
# ============================================================

print("=" * 70)
print("STEP 0: INSTALLING / IMPORTING LIBRARIES")
print("=" * 70)

# Install required packages if needed
!pip -q install kagglehub

import os
import glob
import random
import re
import pickle
import shutil
import time

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

import torchvision.models as models
import torchvision.transforms as transforms

from PIL import Image

# Traditional ML
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve
)

print("✅ Libraries imported successfully.")

STEP 0: INSTALLING / IMPORTING LIBRARIES
✅ Libraries imported successfully.


In [ ]:
# ============================================================
# STEP 1: GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

print("✅ Google Drive mounted.")

Mounted at /content/drive
✅ Google Drive mounted.


In [ ]:
CHECKPOINT_ROOT = (
    "/content/drive/MyDrive/"
    "Handwritten_Signature_Verification"
)

os.makedirs(
    CHECKPOINT_ROOT,
    exist_ok=True
)

print("Checkpoint directory:")
print(CHECKPOINT_ROOT)

Checkpoint directory:
/content/drive/MyDrive/Handwritten_Signature_Verification


In [ ]:
RESNET_CHECKPOINT_DIR = os.path.join(
    CHECKPOINT_ROOT,
    "resnet_checkpoints"
)

os.makedirs(
    RESNET_CHECKPOINT_DIR,
    exist_ok=True
)

print(
    "ResNet checkpoint directory:"
)

print(
    RESNET_CHECKPOINT_DIR
)

ResNet checkpoint directory:
/content/drive/MyDrive/Handwritten_Signature_Verification/resnet_checkpoints


In [ ]:
# ============================================================
# STEP 2: REPRODUCIBILITY
# ============================================================

print("\n" + "=" * 70)
print("STEP 2: SETTING RANDOM SEEDS")
print("=" * 70)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Random seed:", SEED)


STEP 2: SETTING RANDOM SEEDS
Random seed: 42


In [ ]:
# ============================================================
# STEP 3: CONFIGURATION
# ============================================================

print("\n" + "=" * 70)
print("STEP 3: CONFIGURATION")
print("=" * 70)

dataset_root = (
    "/root/.cache/kagglehub/datasets/"
    "ishanikathuria/handwritten-signature-datasets/"
    "versions/3"
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

config = {

    "Dataset Root Path":
        dataset_root,

    "Device":
        str(device),

    "Backbone":
        "ResNet-18",

    "Embedding Dimension":
        128,

    "Batch Size":
        32,

    "Learning Rate":
        0.0001,

    "Weight Decay":
        1e-4,

    "Optimizer":
        "Adam",

    "Loss":
        "Contrastive Loss",

    "Contrastive Margin":
        2.0,

    "Training Epochs":
        5,

    "Image Size":
        (224, 224),

    "Validation Split":
        0.20,

    "Workers":
        2,

    "ML Classifier":
        "SVM",

    # IMPORTANT:
    # sklearn requires lowercase "rbf"
    "SVM Kernel":
        "rbf",

    "SVM C":
        10.0,

    "SVM Gamma":
        "scale"
}

for key, value in config.items():

    print(
        f"{key}: {value}"
    )


STEP 3: CONFIGURATION
Dataset Root Path: /root/.cache/kagglehub/datasets/ishanikathuria/handwritten-signature-datasets/versions/3
Device: cuda
Backbone: ResNet-18
Embedding Dimension: 128
Batch Size: 32
Learning Rate: 0.0001
Weight Decay: 0.0001
Optimizer: Adam
Loss: Contrastive Loss
Contrastive Margin: 2.0
Training Epochs: 5
Image Size: (224, 224)
Validation Split: 0.2
Workers: 2
ML Classifier: SVM
SVM Kernel: rbf
SVM C: 10.0
SVM Gamma: scale


In [ ]:
# ============================================================
# STEP 4: CHECK DATASET
# ============================================================

print("\n" + "=" * 70)
print("STEP 4: CHECKING DATASET")
print("=" * 70)

if not os.path.exists(dataset_root):

    raise FileNotFoundError(
        f"""
Dataset directory does not exist:

{dataset_root}

Please download the BHSig260 dataset first.
"""
    )

print(
    "Dataset directory exists:"
)

print(
    dataset_root
)


STEP 4: CHECKING DATASET
Dataset directory exists:
/root/.cache/kagglehub/datasets/ishanikathuria/handwritten-signature-datasets/versions/3


In [ ]:
# ============================================================
# STEP 5: FIND ALL IMAGES
# ============================================================

print("\n" + "=" * 70)
print("STEP 5: DISCOVERING SIGNATURE IMAGES")
print("=" * 70)

image_extensions = (
    ".png",
    ".jpg",
    ".jpeg",
    ".bmp",
    ".tif",
    ".tiff"
)

all_image_paths = []

for root, dirs, files in os.walk(
    dataset_root
):

    for file in files:

        if file.lower().endswith(
            image_extensions
        ):

            all_image_paths.append(
                os.path.join(
                    root,
                    file
                )
            )

all_image_paths = sorted(
    all_image_paths
)

print(
    f"Total images discovered: "
    f"{len(all_image_paths):,}"
)

if len(all_image_paths) == 0:

    raise RuntimeError(
        "No signature images were found."
    )

print("\nFirst 20 images:")

for path in all_image_paths[:20]:

    print(path)


STEP 5: DISCOVERING SIGNATURE IMAGES
Total images discovered: 16,680

First 20 images:
/root/.cache/kagglehub/datasets/ishanikathuria/handwritten-signature-datasets/versions/3/BHSig260-Bengali/BHSig260-Bengali/1/B-S-1-F-01.tif
/root/.cache/kagglehub/datasets/ishanikathuria/handwritten-signature-datasets/versions/3/BHSig260-Bengali/BHSig260-Bengali/1/B-S-1-F-02.tif
/root/.cache/kagglehub/datasets/ishanikathuria/handwritten-signature-datasets/versions/3/BHSig260-Bengali/BHSig260-Bengali/1/B-S-1-F-03.tif
/root/.cache/kagglehub/datasets/ishanikathuria/handwritten-signature-datasets/versions/3/BHSig260-Bengali/BHSig260-Bengali/1/B-S-1-F-04.tif
/root/.cache/kagglehub/datasets/ishanikathuria/handwritten-signature-datasets/versions/3/BHSig260-Bengali/BHSig260-Bengali/1/B-S-1-F-05.tif
/root/.cache/kagglehub/datasets/ishanikathuria/handwritten-signature-datasets/versions/3/BHSig260-Bengali/BHSig260-Bengali/1/B-S-1-F-06.tif
/root/.cache/kagglehub/datasets/ishanikathuria/handwritten-signature-dat

In [ ]:
# ============================================================
# STEP 6: ANALYZING DATASET STRUCTURE
# ============================================================

print("\n" + "=" * 70)
print("STEP 6: ANALYZING DATASET STRUCTURE")
print("=" * 70)

language_folders = []

for root, dirs, files in os.walk(
    dataset_root
):

    folder_name = os.path.basename(
        root
    ).lower()

    if folder_name in [
        "bengali",
        "hindi",
        "bhsig260-bengali",
        "bhsig260-hindi"
    ]:

        language_folders.append(
            root
        )

print(
    "Detected language folders:",
    len(language_folders)
)

for folder in language_folders:

    print(folder)


STEP 6: ANALYZING DATASET STRUCTURE
Detected language folders: 4
/root/.cache/kagglehub/datasets/ishanikathuria/handwritten-signature-datasets/versions/3/BHSig260-Bengali
/root/.cache/kagglehub/datasets/ishanikathuria/handwritten-signature-datasets/versions/3/BHSig260-Bengali/BHSig260-Bengali
/root/.cache/kagglehub/datasets/ishanikathuria/handwritten-signature-datasets/versions/3/BHSig260-Hindi
/root/.cache/kagglehub/datasets/ishanikathuria/handwritten-signature-datasets/versions/3/BHSig260-Hindi/BHSig260-Hindi


In [ ]:
# ============================================================
# STEP 7: LANGUAGE IDENTIFICATION
# ============================================================

def get_language_from_path(path):

    path_lower = path.lower()

    if "bengali" in path_lower:

        return "bengali"

    if "hindi" in path_lower:

        return "hindi"

    return "unknown"

In [ ]:
# ============================================================
# STEP 8: SIGNER IDENTIFICATION
# ============================================================

def get_signer_id(path):

    """
    Extract signer ID from the folder
    containing the signature image.
    """

    parent_dir = os.path.basename(
        os.path.dirname(path)
    )

    return parent_dir

In [ ]:
# ============================================================
# STEP 9: GENUINE / FORGED IDENTIFICATION
# ============================================================

def classify_signature(path):

    text = path.lower()

    # --------------------------------------------------------
    # Forged keywords
    # --------------------------------------------------------

    forged_keywords = [
        "forg",
        "forged",
        "forgery",
        "forge",
        "fake"
    ]

    # --------------------------------------------------------
    # Genuine keywords
    # --------------------------------------------------------

    genuine_keywords = [
        "genuine",
        "original",
        "real"
    ]

    for keyword in forged_keywords:

        if keyword in text:

            return "forged"

    for keyword in genuine_keywords:

        if keyword in text:

            return "genuine"

    # --------------------------------------------------------
    # Filename based detection
    # --------------------------------------------------------

    filename = os.path.basename(
        path
    ).lower()

    filename_without_ext = os.path.splitext(
        filename
    )[0]

    tokens = re.split(
        r"[_\-\s]+",
        filename_without_ext
    )

    for token in tokens:

        if token in [
            "g",
            "genuine"
        ]:

            return "genuine"

        if token in [
            "f",
            "forged",
            "forgery"
        ]:

            return "forged"

    return "unknown"

In [ ]:
# ============================================================
# STEP 10: GROUP SIGNATURES BY SIGNER
# ============================================================

print("\n" + "=" * 70)
print("STEP 10: GROUPING SIGNATURES BY SIGNER")
print("=" * 70)

signer_images = {}

for image_path in all_image_paths:

    language = get_language_from_path(
        image_path
    )

    signer = get_signer_id(
        image_path
    )

    signature_type = classify_signature(
        image_path
    )

    if signature_type == "unknown":

        continue

    signer_key = (
        language,
        signer
    )

    if signer_key not in signer_images:

        signer_images[signer_key] = {
            "genuine": [],
            "forged": []
        }

    signer_images[
        signer_key
    ][signature_type].append(
        image_path
    )

print(
    "Signers detected:",
    len(signer_images)
)


STEP 10: GROUPING SIGNATURES BY SIGNER
Signers detected: 315


In [ ]:
# ============================================================
# STEP 11: DATASET STATISTICS
# ============================================================

total_genuine = 0
total_forged = 0

for signer_key, data in signer_images.items():

    total_genuine += len(
        data["genuine"]
    )

    total_forged += len(
        data["forged"]
    )

print(
    f"Genuine signatures: "
    f"{total_genuine:,}"
)

print(
    f"Forged signatures: "
    f"{total_forged:,}"
)

if len(signer_images) == 0:

    raise RuntimeError(
        """
Dataset structure could not be identified.

Please inspect the printed image paths.
"""
    )

Genuine signatures: 7,560
Forged signatures: 9,120


In [ ]:
# ============================================================
# STEP 12: TRAIN / VALIDATION SPLIT
# ============================================================

print("\n" + "=" * 70)
print("STEP 12: TRAIN / VALIDATION SPLIT")
print("=" * 70)

signer_keys = list(
    signer_images.keys()
)

random.shuffle(
    signer_keys
)

split_index = int(
    len(signer_keys)
    *
    (
        1 -
        config["Validation Split"]
    )
)

train_signers = signer_keys[
    :split_index
]

val_signers = signer_keys[
    split_index:
]

print(
    "Training signers:",
    len(train_signers)
)

print(
    "Validation signers:",
    len(val_signers)
)


STEP 12: TRAIN / VALIDATION SPLIT
Training signers: 252
Validation signers: 63


In [ ]:
# ============================================================
# STEP 13: CREATE SIGNATURE PAIRS
# ============================================================

def create_pairs(
    signer_list,
    max_positive_per_signer=100,
    max_negative_per_signer=100
):

    pairs = []
    labels = []

    for signer_key in signer_list:

        genuine = signer_images[
            signer_key
        ]["genuine"]

        forged = signer_images[
            signer_key
        ]["forged"]

        # ----------------------------------------------------
        # POSITIVE PAIRS
        # Genuine + Genuine
        # Label = 0
        # ----------------------------------------------------

        positive_pairs = []

        for i in range(
            len(genuine)
        ):

            for j in range(
                i + 1,
                len(genuine)
            ):

                positive_pairs.append(
                    (
                        genuine[i],
                        genuine[j]
                    )
                )

        random.shuffle(
            positive_pairs
        )

        positive_pairs = positive_pairs[
            :max_positive_per_signer
        ]

        for pair in positive_pairs:

            pairs.append(pair)

            labels.append(
                0.0
            )

        # ----------------------------------------------------
        # NEGATIVE PAIRS
        # Genuine + Forged
        # Label = 1
        # ----------------------------------------------------

        negative_pairs = []

        for genuine_path in genuine:

            for forged_path in forged:

                negative_pairs.append(
                    (
                        genuine_path,
                        forged_path
                    )
                )

        random.shuffle(
            negative_pairs
        )

        negative_pairs = negative_pairs[
            :max_negative_per_signer
        ]

        for pair in negative_pairs:

            pairs.append(pair)

            labels.append(
                1.0
            )

    return pairs, labels

In [ ]:
# ============================================================
# STEP 14: CREATE TRAINING / VALIDATION PAIRS
# ============================================================

print("\n" + "=" * 70)
print("STEP 14: GENERATING SIGNATURE PAIRS")
print("=" * 70)

train_pairs, train_labels = create_pairs(
    train_signers
)

val_pairs, val_labels = create_pairs(
    val_signers
)

print(
    "\nTraining pairs:"
)

print(
    "Genuine-Genuine:",
    train_labels.count(0.0)
)

print(
    "Genuine-Forged:",
    train_labels.count(1.0)
)

print(
    "Total:",
    len(train_pairs)
)

print(
    "\nValidation pairs:"
)

print(
    "Genuine-Genuine:",
    val_labels.count(0.0)
)

print(
    "Genuine-Forged:",
    val_labels.count(1.0)
)

print(
    "Total:",
    len(val_pairs)
)

if len(train_pairs) == 0:

    raise RuntimeError(
        """
Training pairs are zero.

Check the genuine/forged dataset classification.
"""
    )


STEP 14: GENERATING SIGNATURE PAIRS

Training pairs:
Genuine-Genuine: 25200
Genuine-Forged: 25200
Total: 50400

Validation pairs:
Genuine-Genuine: 6300
Genuine-Forged: 6300
Total: 12600


In [ ]:
# ============================================================
# STEP 15: IMAGE PREPROCESSING
# ============================================================

print("\n" + "=" * 70)
print("STEP 15: IMAGE PREPROCESSING")
print("=" * 70)

train_transform = transforms.Compose([

    transforms.Resize(
        config["Image Size"]
    ),

    transforms.RandomRotation(
        degrees=5
    ),

    transforms.RandomAffine(
        degrees=0,
        translate=(0.03, 0.03),
        scale=(0.95, 1.05)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],

        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])

val_transform = transforms.Compose([

    transforms.Resize(
        config["Image Size"]
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],

        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])

print("✅ Image transforms created.")


STEP 15: IMAGE PREPROCESSING
✅ Image transforms created.


In [ ]:
# ============================================================
# STEP 16: PYTORCH DATASET
# ============================================================

class SignaturePairDataset(
    Dataset
):

    def __init__(
        self,
        pairs,
        labels,
        transform=None
    ):

        self.pairs = pairs
        self.labels = labels
        self.transform = transform

    def __len__(self):

        return len(
            self.pairs
        )

    def __getitem__(
        self,
        index
    ):

        image1_path, image2_path = (
            self.pairs[index]
        )

        image1 = Image.open(
            image1_path
        ).convert(
            "RGB"
        )

        image2 = Image.open(
            image2_path
        ).convert(
            "RGB"
        )

        if self.transform:

            image1 = self.transform(
                image1
            )

            image2 = self.transform(
                image2
            )

        label = torch.tensor(
            self.labels[index],
            dtype=torch.float32
        )

        return (
            image1,
            image2,
            label
        )

In [ ]:
# ============================================================
# STEP 17: DATA LOADERS
# ============================================================

train_dataset = SignaturePairDataset(
    train_pairs,
    train_labels,
    transform=train_transform
)

val_dataset = SignaturePairDataset(
    val_pairs,
    val_labels,
    transform=val_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=config["Batch Size"],
    shuffle=True,
    num_workers=config["Workers"],
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config["Batch Size"],
    shuffle=False,
    num_workers=config["Workers"],
    pin_memory=torch.cuda.is_available()
)

print(
    "Training batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(val_loader)
)

Training batches: 1575
Validation batches: 394


In [ ]:
# ============================================================
# STEP 18: SIAMESE RESNET-18
# ============================================================

print("\n" + "=" * 70)
print("STEP 18: BUILDING SIAMESE RESNET-18")
print("=" * 70)

class SiameseResNet(
    nn.Module
):

    def __init__(
        self,
        embedding_dim=128
    ):

        super().__init__()

        # ----------------------------------------------------
        # Pretrained ResNet18
        # ----------------------------------------------------

        self.backbone = models.resnet18(
            weights=
            models.ResNet18_Weights.DEFAULT
        )

        num_features = (
            self.backbone.fc.in_features
        )

        self.backbone.fc = nn.Identity()

        # ----------------------------------------------------
        # Embedding head
        # ----------------------------------------------------

        self.fc_head = nn.Sequential(

            nn.Linear(
                num_features,
                512
            ),

            nn.BatchNorm1d(
                512
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Dropout(
                p=0.3
            ),

            nn.Linear(
                512,
                embedding_dim
            )
        )

    def forward_once(
        self,
        x
    ):

        features = self.backbone(
            x
        )

        embedding = self.fc_head(
            features
        )

        embedding = nn.functional.normalize(
            embedding,
            p=2,
            dim=1
        )

        return embedding

    def forward(
        self,
        input1,
        input2
    ):

        output1 = self.forward_once(
            input1
        )

        output2 = self.forward_once(
            input2
        )

        return (
            output1,
            output2
        )


model = SiameseResNet(
    embedding_dim=
    config["Embedding Dimension"]
).to(device)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    "Model created."
)

print(
    "Total parameters:",
    f"{total_params:,}"
)

print(
    "Trainable parameters:",
    f"{trainable_params:,}"
)


STEP 18: BUILDING SIAMESE RESNET-18
Model created.
Total parameters: 11,505,856
Trainable parameters: 11,505,856


In [ ]:
# ============================================================
# STEP 19: CONTRASTIVE LOSS
# ============================================================

class ContrastiveLoss(
    nn.Module
):

    def __init__(
        self,
        margin=2.0
    ):

        super().__init__()

        self.margin = margin

    def forward(
        self,
        output1,
        output2,
        label
    ):

        distance = nn.functional.pairwise_distance(
            output1,
            output2
        )

        positive_loss = (
            (1 - label)
            *
            torch.pow(
                distance,
                2
            )
        )

        negative_loss = (
            label
            *
            torch.pow(
                torch.clamp(
                    self.margin -
                    distance,
                    min=0.0
                ),
                2
            )
        )

        loss = torch.mean(
            positive_loss +
            negative_loss
        )

        return loss


criterion = ContrastiveLoss(
    margin=config[
        "Contrastive Margin"
    ]
)

optimizer = optim.Adam(
    model.parameters(),
    lr=config[
        "Learning Rate"
    ],
    weight_decay=config[
        "Weight Decay"
    ]
)

print("✅ Loss and optimizer created.")

✅ Loss and optimizer created.


In [ ]:
# ============================================================
# STEP 20: VALIDATION FUNCTION
# ============================================================

def evaluate_siamese_model(
    model,
    data_loader,
    criterion,
    device,
    threshold=1.0
):

    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for img1, img2, label in data_loader:

            img1 = img1.to(
                device,
                non_blocking=True
            )

            img2 = img2.to(
                device,
                non_blocking=True
            )

            label = label.to(
                device,
                non_blocking=True
            )

            output1, output2 = model(
                img1,
                img2
            )

            loss = criterion(
                output1,
                output2,
                label
            )

            total_loss += (
                loss.item()
                *
                img1.size(0)
            )

            distance = nn.functional.pairwise_distance(
                output1,
                output2
            )

            predicted = (
                distance > threshold
            ).float()

            correct += (
                predicted == label
            ).sum().item()

            total += label.size(0)

    avg_loss = (
        total_loss / total
        if total > 0
        else 0
    )

    accuracy = (
        correct / total
        if total > 0
        else 0
    )

    return avg_loss, accuracy

In [ ]:
# ============================================================
# STEP 21: TRAIN SIAMESE RESNET-18 WITH CHECKPOINTS
# ============================================================

print("\n" + "=" * 70)
print("STEP 21: TRAINING SIAMESE RESNET-18")
print("=" * 70)

latest_checkpoint = os.path.join(
    RESNET_CHECKPOINT_DIR,
    "checkpoint_latest.pth"
)

start_epoch = 0

best_val_loss = float("inf")


# ============================================================
# CHECK FOR EXISTING CHECKPOINT
# ============================================================

if os.path.exists(
    latest_checkpoint
):

    print("\n" + "=" * 70)
    print("EXISTING CHECKPOINT FOUND")
    print("=" * 70)

    print(
        latest_checkpoint
    )

    checkpoint = torch.load(
        latest_checkpoint,
        map_location=device
    )

    model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ]
    )

    optimizer.load_state_dict(
        checkpoint[
            "optimizer_state_dict"
        ]
    )

    start_epoch = (
        checkpoint["epoch"] + 1
    )

    best_val_loss = checkpoint[
        "best_val_loss"
    ]

    print(
        "\n✅ Model restored."
    )

    print(
        "Last completed epoch:",
        checkpoint["epoch"] + 1
    )

    print(
        "Next epoch:",
        start_epoch + 1
    )

    print(
        "Best validation loss:",
        f"{best_val_loss:.4f}"
    )

else:

    print(
        "\nNo previous checkpoint found."
    )

    print(
        "Starting training from epoch 1."
    )


# ============================================================
# TRAINING LOOP
# ============================================================

for epoch in range(
    start_epoch,
    config["Training Epochs"]
):

    model.train()

    running_loss = 0.0

    print("\n" + "-" * 70)

    print(
        f"Epoch [{epoch + 1}/"
        f"{config['Training Epochs']}]"
    )

    print("-" * 70)


    # --------------------------------------------------------
    # TRAINING
    # --------------------------------------------------------

    for batch_idx, (
        img1,
        img2,
        label
    ) in enumerate(
        train_loader
    ):

        img1 = img1.to(
            device,
            non_blocking=True
        )

        img2 = img2.to(
            device,
            non_blocking=True
        )

        label = label.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad()

        output1, output2 = model(
            img1,
            img2
        )

        loss = criterion(
            output1,
            output2,
            label
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()


        # ----------------------------------------------------
        # PRINT PROGRESS
        # ----------------------------------------------------

        if (
            (batch_idx + 1) % 20 == 0
            or
            (batch_idx + 1)
            ==
            len(train_loader)
        ):

            print(
                f"Batch "
                f"[{batch_idx + 1}/"
                f"{len(train_loader)}] "
                f"Loss: "
                f"{loss.item():.4f}"
            )


    # --------------------------------------------------------
    # TRAIN LOSS
    # --------------------------------------------------------

    train_loss = (
        running_loss /
        len(train_loader)
    )


    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    val_loss, val_accuracy = (
        evaluate_siamese_model(
            model,
            val_loader,
            criterion,
            device,
            threshold=1.0
        )
    )


    print(
        f"\nTrain Loss: "
        f"{train_loss:.4f}"
    )

    print(
        f"Validation Loss: "
        f"{val_loss:.4f}"
    )

    print(
        f"Validation Accuracy: "
        f"{val_accuracy * 100:.2f}%"
    )


    # ========================================================
    # CHECK WHETHER THIS IS BEST MODEL
    # ========================================================

    is_best = (
        val_loss < best_val_loss
    )

    if is_best:

        best_val_loss = val_loss

        print(
            "\n⭐ New best validation loss!"
        )


    # ========================================================
    # CREATE CHECKPOINT
    # ========================================================

    checkpoint = {

        "epoch":
            epoch,

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "train_loss":
            train_loss,

        "val_loss":
            val_loss,

        "val_accuracy":
            val_accuracy,

        "best_val_loss":
            best_val_loss,

        "config":
            config,

        "random_state":
            random.getstate(),

        "numpy_random_state":
            np.random.get_state()
    }


    # ========================================================
    # SAVE LATEST CHECKPOINT
    # ========================================================

    torch.save(
        checkpoint,
        latest_checkpoint
    )

    print(
        "\n💾 Latest checkpoint saved:"
    )

    print(
        latest_checkpoint
    )


    # ========================================================
    # SAVE EPOCH CHECKPOINT
    # ========================================================

    epoch_checkpoint = os.path.join(
        RESNET_CHECKPOINT_DIR,
        f"checkpoint_epoch_{epoch + 1}.pth"
    )

    torch.save(
        checkpoint,
        epoch_checkpoint
    )

    print(
        "💾 Epoch checkpoint saved:"
    )

    print(
        epoch_checkpoint
    )


    # ========================================================
    # SAVE BEST MODEL
    # ========================================================

    if is_best:

        best_model_path = os.path.join(
            CHECKPOINT_ROOT,
            "best_siamese_resnet_signature.pth"
        )

        torch.save(
            model.state_dict(),
            best_model_path
        )

        print(
            "⭐ Best Siamese model saved:"
        )

        print(
            best_model_path
        )


print("\n" + "=" * 70)

print(
    "SIAMESE RESNET TRAINING COMPLETED"
)

print("=" * 70)


STEP 21: TRAINING SIAMESE RESNET-18

No previous checkpoint found.
Starting training from epoch 1.

----------------------------------------------------------------------
Epoch [1/5]
----------------------------------------------------------------------
Batch [20/1575] Loss: 0.8963
Batch [40/1575] Loss: 0.8354
Batch [60/1575] Loss: 0.8036
Batch [80/1575] Loss: 0.5363
Batch [100/1575] Loss: 0.5711
Batch [120/1575] Loss: 0.3903
Batch [140/1575] Loss: 0.5113
Batch [160/1575] Loss: 0.4739
Batch [180/1575] Loss: 0.4272
Batch [200/1575] Loss: 0.3154
Batch [220/1575] Loss: 0.4573
Batch [240/1575] Loss: 0.3282
Batch [260/1575] Loss: 0.3766
Batch [280/1575] Loss: 0.2598
Batch [300/1575] Loss: 0.2826
Batch [320/1575] Loss: 0.2695
Batch [340/1575] Loss: 0.2778
Batch [360/1575] Loss: 0.1865
Batch [380/1575] Loss: 0.4676
Batch [400/1575] Loss: 0.3010
Batch [420/1575] Loss: 0.1525
Batch [440/1575] Loss: 0.4442
Batch [460/1575] Loss: 0.1756
Batch [480/1575] Loss: 0.1633
Batch [500/1575] Loss: 0.1514

In [ ]:
# ============================================================
# STEP 22: CHECKPOINT FILES
# ============================================================

print("\n" + "=" * 70)
print("STEP 22: SAVED CHECKPOINTS")
print("=" * 70)

checkpoint_files = sorted(
    glob.glob(
        os.path.join(
            RESNET_CHECKPOINT_DIR,
            "*.pth"
        )
    )
)

for file in checkpoint_files:

    size_mb = (
        os.path.getsize(file)
        /
        (1024 * 1024)
    )

    print(
        f"{os.path.basename(file):45s}"
        f"{size_mb:10.2f} MB"
    )

In [ ]:
# ============================================================
# STEP 23: LOAD BEST SIAMESE MODEL
# ============================================================

print("\n" + "=" * 70)
print("STEP 23: LOADING BEST SIAMESE MODEL")
print("=" * 70)

best_model_path = os.path.join(
    CHECKPOINT_ROOT,
    "best_siamese_resnet_signature.pth"
)

if os.path.exists(
    best_model_path
):

    model.load_state_dict(
        torch.load(
            best_model_path,
            map_location=device
        )
    )

    print(
        "✅ Best model loaded successfully."
    )

else:

    print(
        "⚠️ Best model not found."
    )

model.eval()

In [ ]:
# ============================================================
# STEP 24: SAVE FINAL SIAMESE MODEL
# ============================================================

print("\n" + "=" * 70)
print("STEP 24: SAVING FINAL SIAMESE MODEL")
print("=" * 70)

final_model_path = os.path.join(
    CHECKPOINT_ROOT,
    "siamese_resnet_signature_final.pth"
)

torch.save(
    model.state_dict(),
    final_model_path
)

print(
    "Final Siamese model saved:"
)

print(
    final_model_path
)

In [ ]:
# ============================================================
# STEP 25: IMAGE LOADING
# ============================================================

def load_single_image(
    image_path,
    transform
):

    image = Image.open(
        image_path
    ).convert(
        "RGB"
    )

    image = transform(
        image
    )

    image = image.unsqueeze(
        0
    )

    return image

In [ ]:
# ============================================================
# STEP 26: EXTRACT EMBEDDING
# ============================================================

def extract_embedding(
    model,
    image_path,
    transform,
    device
):

    model.eval()

    image = load_single_image(
        image_path,
        transform
    ).to(device)

    with torch.no_grad():

        embedding = model.forward_once(
            image
        )

    embedding = (
        embedding
        .cpu()
        .numpy()[0]
    )

    return embedding

In [ ]:
# ============================================================
# STEP 27: CREATE ML CLASSIFICATION FEATURES
# ============================================================

print("\n" + "=" * 70)
print("STEP 27: CREATING ML CLASSIFICATION FEATURES")
print("=" * 70)

def create_pair_features(
    embedding1,
    embedding2
):

    absolute_difference = np.abs(
        embedding1 -
        embedding2
    )

    multiplication = (
        embedding1 *
        embedding2
    )

    features = np.concatenate(
        [
            absolute_difference,
            multiplication
        ]
    )

    return features


print(
    "Feature type:"
)

print(
    "Absolute Difference + "
    "Element-wise Multiplication"
)

print(
    "Features per pair: 256"
)

In [ ]:
# ============================================================
# STEP 28: GENERATE ML FEATURES
# ============================================================

def generate_ml_features(
    model,
    pairs,
    labels,
    transform,
    device
):

    X = []
    y = []

    print(
        f"Generating ML features "
        f"for {len(pairs):,} pairs..."
    )

    for index, (
        image1_path,
        image2_path
    ) in enumerate(
        pairs
    ):

        embedding1 = extract_embedding(
            model,
            image1_path,
            transform,
            device
        )

        embedding2 = extract_embedding(
            model,
            image2_path,
            transform,
            device
        )

        features = create_pair_features(
            embedding1,
            embedding2
        )

        X.append(
            features
        )

        y.append(
            labels[index]
        )

        if (
            (index + 1) % 500 == 0
            or
            (index + 1) == len(pairs)
        ):

            print(
                f"Processed "
                f"{index + 1:,}/"
                f"{len(pairs):,}"
            )

    X = np.array(
        X,
        dtype=np.float32
    )

    y = np.array(
        y,
        dtype=np.int64
    )

    return X, y

In [ ]:
# ============================================================
# STEP 29: GENERATE TRAINING ML FEATURES
# ============================================================

print("\n" + "=" * 70)
print("STEP 29: GENERATING TRAINING FEATURES")
print("=" * 70)

X_train_ml, y_train_ml = generate_ml_features(
    model,
    train_pairs,
    train_labels,
    val_transform,
    device
)

print(
    "\nTraining feature matrix:"
)

print(
    "Shape:",
    X_train_ml.shape
)

print(
    "Training labels:",
    y_train_ml.shape
)


STEP 29: GENERATING TRAINING FEATURES
Generating ML features for 50,400 pairs...
Processed 500/50,400
Processed 1,000/50,400
Processed 1,500/50,400
Processed 2,000/50,400
Processed 2,500/50,400
Processed 3,000/50,400
Processed 3,500/50,400
Processed 4,000/50,400
Processed 4,500/50,400
Processed 5,000/50,400
Processed 5,500/50,400
Processed 6,000/50,400
Processed 6,500/50,400
Processed 7,000/50,400
Processed 7,500/50,400
Processed 8,000/50,400
Processed 8,500/50,400
Processed 9,000/50,400
Processed 9,500/50,400
Processed 10,000/50,400
Processed 10,500/50,400
Processed 11,000/50,400
Processed 11,500/50,400
Processed 12,000/50,400
Processed 12,500/50,400
Processed 13,000/50,400
Processed 13,500/50,400
Processed 14,000/50,400
Processed 14,500/50,400
Processed 15,000/50,400
Processed 15,500/50,400
Processed 16,000/50,400
Processed 16,500/50,400
Processed 17,000/50,400
Processed 17,500/50,400
Processed 18,000/50,400
Processed 18,500/50,400
Processed 19,000/50,400
Processed 19,500/50,400
Pro

In [ ]:
# ============================================================
# STEP 30: GENERATE VALIDATION ML FEATURES
# ============================================================

print("\n" + "=" * 70)
print("STEP 30: GENERATING VALIDATION FEATURES")
print("=" * 70)

X_val_ml, y_val_ml = generate_ml_features(
    model,
    val_pairs,
    val_labels,
    val_transform,
    device
)

print(
    "\nValidation feature matrix:"
)

print(
    "Shape:",
    X_val_ml.shape
)

print(
    "Validation labels:",
    y_val_ml.shape
)


STEP 30: GENERATING VALIDATION FEATURES
Generating ML features for 12,600 pairs...
Processed 500/12,600
Processed 1,000/12,600
Processed 1,500/12,600
Processed 2,000/12,600
Processed 2,500/12,600
Processed 3,000/12,600
Processed 3,500/12,600
Processed 4,000/12,600
Processed 4,500/12,600
Processed 5,000/12,600
Processed 5,500/12,600
Processed 6,000/12,600
Processed 6,500/12,600
Processed 7,000/12,600
Processed 7,500/12,600
Processed 8,000/12,600
Processed 8,500/12,600
Processed 9,000/12,600
Processed 9,500/12,600
Processed 10,000/12,600
Processed 10,500/12,600
Processed 11,000/12,600
Processed 11,500/12,600
Processed 12,000/12,600
Processed 12,500/12,600
Processed 12,600/12,600

Validation feature matrix:
Shape: (12600, 256)
Validation labels: (12600,)


In [ ]:
# ============================================================
# STEP 31: FEATURE STANDARDIZATION
# ============================================================

print("\n" + "=" * 70)
print("STEP 31: FEATURE STANDARDIZATION")
print("=" * 70)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train_ml
)

X_val_scaled = scaler.transform(
    X_val_ml
)

print(
    "StandardScaler fitted on training data."
)

print(
    "Training shape:",
    X_train_scaled.shape
)

print(
    "Validation shape:",
    X_val_scaled.shape
)


STEP 31: FEATURE STANDARDIZATION
StandardScaler fitted on training data.
Training shape: (50400, 256)
Validation shape: (12600, 256)


In [ ]:
kernel="rbf"

In [ ]:
# ============================================================
# STEP 32: TRAIN SVM CLASSIFIER
# ============================================================

print("\n" + "=" * 70)
print("STEP 32: TRAINING MACHINE LEARNING CLASSIFIER")
print("=" * 70)

print(
    "Algorithm: Support Vector Machine"
)

print(
    "Kernel:",
    config["SVM Kernel"]
)

print(
    "C:",
    config["SVM C"]
)

print(
    "Gamma:",
    config["SVM Gamma"]
)

svm_classifier = SVC(
    kernel=config["SVM Kernel"],
    C=config["SVM C"],
    gamma=config["SVM Gamma"],
    probability=True,
    random_state=SEED
)

svm_classifier.fit(
    X_train_scaled,
    y_train_ml
)

print(
    "\n✅ SVM training completed successfully."
)

In [ ]:
# ============================================================
# STEP 33: SVM MODEL EVALUATION
# ============================================================

print("\n" + "=" * 70)
print("STEP 33: SVM MODEL EVALUATION")
print("=" * 70)

y_pred = svm_classifier.predict(
    X_val_scaled
)

y_probability = svm_classifier.predict_proba(
    X_val_scaled
)[:, 1]


STEP 33: SVM MODEL EVALUATION


In [ ]:
# ============================================================
# STEP 34: CLASSIFICATION METRICS
# ============================================================

accuracy = accuracy_score(
    y_val_ml,
    y_pred
)

precision = precision_score(
    y_val_ml,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_val_ml,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_val_ml,
    y_pred,
    zero_division=0
)

roc_auc = roc_auc_score(
    y_val_ml,
    y_probability
)

print(
    "\n=============================="
)

print(
    "SVM CLASSIFICATION RESULTS"
)

print(
    "=============================="
)

print(
    f"Accuracy  : {accuracy * 100:.2f}%"
)

print(
    f"Precision : {precision * 100:.2f}%"
)

print(
    f"Recall    : {recall * 100:.2f}%"
)

print(
    f"F1 Score  : {f1 * 100:.2f}%"
)

print(
    f"ROC-AUC   : {roc_auc:.4f}"
)


SVM CLASSIFICATION RESULTS
Accuracy  : 93.65%
Precision : 92.98%
Recall    : 94.43%
F1 Score  : 93.70%
ROC-AUC   : 0.9755


In [ ]:
# ============================================================
# STEP 35: CLASSIFICATION REPORT
# ============================================================

print("\n" + "=" * 70)
print("STEP 35: CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        y_val_ml,
        y_pred,
        target_names=[
            "Genuine",
            "Forged"
        ],
        zero_division=0
    )
)


STEP 35: CLASSIFICATION REPORT
              precision    recall  f1-score   support

     Genuine       0.94      0.93      0.94      6300
      Forged       0.93      0.94      0.94      6300

    accuracy                           0.94     12600
   macro avg       0.94      0.94      0.94     12600
weighted avg       0.94      0.94      0.94     12600



In [ ]:
# ============================================================
# STEP 36: CONFUSION MATRIX
# ============================================================

print("\n" + "=" * 70)
print("STEP 36: CONFUSION MATRIX")
print("=" * 70)

cm = confusion_matrix(
    y_val_ml,
    y_pred
)

print(cm)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        "Genuine",
        "Forged"
    ]
)

disp.plot(
    values_format="d"
)

plt.title(
    "SVM - Signature Verification Confusion Matrix"
)

plt.show()

In [ ]:
# ============================================================
# STEP 37: ROC CURVE
# ============================================================

print("\n" + "=" * 70)
print("STEP 37: ROC CURVE")
print("=" * 70)

fpr, tpr, thresholds = roc_curve(
    y_val_ml,
    y_probability
)

plt.figure(
    figsize=(7, 5)
)

plt.plot(
    fpr,
    tpr,
    label=f"ROC-AUC = {roc_auc:.4f}"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--"
)

plt.xlabel(
    "False Positive Rate"
)

plt.ylabel(
    "True Positive Rate"
)

plt.title(
    "SVM ROC Curve - Signature Verification"
)

plt.legend()

plt.grid()

plt.show()

In [ ]:
# ============================================================
# STEP 38: SAVE SVM MODEL
# ============================================================

print("\n" + "=" * 70)
print("STEP 38: SAVING MACHINE LEARNING MODEL")
print("=" * 70)

svm_model_path = os.path.join(
    CHECKPOINT_ROOT,
    "signature_svm_classifier.pkl"
)

scaler_path = os.path.join(
    CHECKPOINT_ROOT,
    "signature_feature_scaler.pkl"
)

with open(
    svm_model_path,
    "wb"
) as file:

    pickle.dump(
        svm_classifier,
        file
    )

with open(
    scaler_path,
    "wb"
) as file:

    pickle.dump(
        scaler,
        file
    )

print(
    "SVM model saved:"
)

print(
    svm_model_path
)

print(
    "\nScaler saved:"
)

print(
    scaler_path
)

In [ ]:
# ============================================================
# STEP 39: FINAL SIGNATURE VERIFICATION FUNCTION
# ============================================================

print("\n" + "=" * 70)
print("STEP 39: FINAL SIGNATURE VERIFICATION FUNCTION")
print("=" * 70)

def verify_signature(
    image1_path,
    image2_path
):

    """
    Compare two signature images.

    Returns:
        Genuine / Forged
        Confidence
    """

    # --------------------------------------------------------
    # Extract embeddings
    # --------------------------------------------------------

    embedding1 = extract_embedding(
        model,
        image1_path,
        val_transform,
        device
    )

    embedding2 = extract_embedding(
        model,
        image2_path,
        val_transform,
        device
    )

    # --------------------------------------------------------
    # Create ML feature vector
    # --------------------------------------------------------

    features = create_pair_features(
        embedding1,
        embedding2
    )

    features = features.reshape(
        1,
        -1
    )

    # --------------------------------------------------------
    # Standardize
    # --------------------------------------------------------

    features_scaled = scaler.transform(
        features
    )

    # --------------------------------------------------------
    # SVM prediction
    # --------------------------------------------------------

    prediction = svm_classifier.predict(
        features_scaled
    )[0]

    probability = svm_classifier.predict_proba(
        features_scaled
    )[0]

    # --------------------------------------------------------
    # Result
    # --------------------------------------------------------

    if prediction == 0:

        result = "GENUINE"

    else:

        result = "FORGED"

    confidence = (
        probability[prediction]
        *
        100
    )

    print("\n" + "=" * 60)

    print(
        "SIGNATURE VERIFICATION RESULT"
    )

    print("=" * 60)

    print(
        "Prediction:",
        result
    )

    print(
        f"Confidence: "
        f"{confidence:.2f}%"
    )

    print("=" * 60)

    return result, confidence

In [ ]:
# ============================================================
# STEP 40: TESTING
# ============================================================

print("\n" + "=" * 70)
print("STEP 40: TESTING")
print("=" * 70)

print(
    """
Upload two signature images.

Then run:

result, confidence = verify_signature(
    "/content/signature1.png",
    "/content/signature2.png"
)
"""
)

In [ ]:
# ============================================================
# STEP 41: PROJECT SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("STEP 41: PROJECT SUMMARY")
print("=" * 70)

print(
    """
============================================================
HANDWRITTEN SIGNATURE VERIFICATION USING ML
============================================================

Deep Learning:
    Siamese ResNet-18

Embedding:
    128-dimensional signature embedding

Feature Engineering:
    Absolute embedding difference
    Element-wise embedding multiplication

Traditional Machine Learning:
    Support Vector Machine (SVM)
    Kernel: RBF

Classification:
    0 = Genuine
    1 = Forged

Evaluation:
    Accuracy
    Precision
    Recall
    F1-score
    ROC-AUC
    Confusion Matrix
    ROC Curve

Checkpointing:
    checkpoint_latest.pth
    checkpoint_epoch_1.pth
    checkpoint_epoch_2.pth
    ...

Saved Models:
    best_siamese_resnet_signature.pth
    siamese_resnet_signature_final.pth
    signature_svm_classifier.pkl
    signature_feature_scaler.pkl

Final Pipeline:

    Signature Image 1
           +
    Signature Image 2
           ↓
    Siamese ResNet-18
           ↓
    128-D Embeddings
           ↓
    Pair Feature Engineering
           ↓
    256 Features
           ↓
    StandardScaler
           ↓
    RBF SVM
           ↓
    Genuine / Forged
============================================================
"""
)

print(
    "\n🎉 COMPLETE ML-BASED SIGNATURE "
    "VERIFICATION PIPELINE FINISHED!"
)

In [57]:

svm_model = load_svm()

NameError: name 'load_svm' is not defined